In [ ]:
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# 1. KHỞI TẠO SPARKSESSION
spark = SparkSession.builder \
    .appName("Weather_Preprocessing_Jupyter") \
    .master("local[*]") \
    .config("spark.driver.memory", "28g") \
    .config("spark.executor.memory", "28g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()
# 2. ĐỌC DỮ LIỆU ĐÃ TIỀN XỬ LÝ
train_data = spark.read.parquet("train_data_folderkhongchuan.parquet")
test_data = spark.read.parquet("test_data_folderkhongchuan.parquet")
# 3. TẠO TẬP VALIDATION
print("⏳ Đang chuẩn bị tập Validation...")
# Chia tập train thành 80% để học thực sự và 20% để kiểm tra trong lúc học
train_sub, val_sub = train_data.randomSplit([0.8, 0.2], seed=42)
# Thêm cột 'is_val' để đánh dấu: False là Train, True là Validation
train_sub = train_sub.withColumn("is_val", F.lit(False))
val_sub = val_sub.withColumn("is_val", F.lit(True))
# Gộp lại thành một DataFrame tổng duy nhất để đưa vào mô hình
train_with_val = train_sub.union(val_sub)
# -------------------------------------------------------------------------
# 4. KHỞI TẠO VÀ HUẤN LUYỆN XGBOOST
xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="rain_label",
    num_workers=8,
    n_estimators=10000,
    max_depth=6,
    learning_rate=0.02,
    early_stopping_rounds=20,
    validation_indicator_col="is_val",
    scale_pos_weight=6
)
print("🚀 Đang tiến hành huấn luyện XGBoost (có Early Stopping)...")
xgb_model = xgb_classifier.fit(train_with_val)
print("✅ Huấn luyện thành công!")
# 5. DỰ ĐOÁN VÀ ĐÁNH GIÁ TRÊN TẬP TEST GỐC
print("🔮 Đang dự đoán trên tập Test...")
predictions = xgb_model.transform(test_data)
# 6. LƯU MÔ HÌNH
model_path = "xgboost_weather_model_kosmotekhongchuan"
xgb_model.write().overwrite().save(model_path)
print(f"💾 Đã lưu mô hình thành công tại thư mục: {model_path}")
# 7. ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH
evaluator = MulticlassClassificationEvaluator(labelCol="rain_label", predictionCol="prediction")
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision_rain = evaluator.evaluate(predictions, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1.0})
recall_rain = evaluator.evaluate(predictions, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1.0})
f1_rain = evaluator.evaluate(predictions, {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: 1.0})

# Tạo nội dung báo cáo
report_content = f"""
==================================================
📊 BẢNG TỔNG HỢP CHỈ SỐ ĐÁNH GIÁ (SAU KHI HYBRID RESAMPLING)
==================================================
✔️ Accuracy  : {accuracy * 100:.2f}%
🎯 Precision : {precision_rain * 100:.2f}%
🎯 Recall    : {recall_rain * 100:.2f}%
🏆 F1-Score  : {f1_rain * 100:.2f}%
==================================================
"""

# In ra màn hình console như bình thường
print(report_content)

# Lưu nội dung vào file text
output_file = "evaluation_results_kosmotekhongchuan.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"📁 Đã xuất kết quả đánh giá thành công ra file: {output_file}")

# --- Hướng dẫn cách Load lại mô hình sau này ---
# from xgboost.spark import SparkXGBClassifierModel
# loaded_model = SparkXGBClassifierModel.load(model_path)
# predictions_new = loaded_model.transform(new_data)

In [ ]:
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# 1. Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Weather_XGBoost_Final") \
    .master("local[*]") \
    .config("spark.driver.memory", "28g") \
    .config("spark.executor.memory", "28g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()
# 2. ĐỌC DỮ LIỆU ĐÃ TIỀN XỬ LÝ
train_data = spark.read.parquet("train_data_folderkhongchuan.parquet")
test_data = spark.read.parquet("test_data_folderkhongchuan.parquet")
# 3. SMOTE
df_nang = train_data.filter(F.col("rain_label") == 0)
df_mua = train_data.filter(F.col("rain_label") == 1)
so_nang_cu = df_nang.count()
so_mua_cu = df_mua.count()
# Giảm 2 lần nắng
df_nang_giam = df_nang.sample(withReplacement=False, fraction=0.5, seed=42)
# Tăng 2 lần mưa
df_mua_tang = df_mua.sample(withReplacement=True, fraction=3.0, seed=42)
# Gộp 2 tập mới lại và xáo trộn (Shuffle)
train_hybrid = df_nang_giam.unionAll(df_mua_tang).orderBy(F.rand(seed=42))
# Đếm lại số lượng để tính tỷ lệ bập bênh mới
so_nang_moi = df_nang_giam.count()
so_mua_moi = df_mua_tang.count()
ty_le_lech_moi = so_nang_moi / so_mua_moi
print(f"📊 Tỷ lệ ban đầu -> Nắng: {so_nang_cu} | Mưa: {so_mua_cu}")
print(f"📊 Tỷ lệ lúc sau  -> Nắng: {so_nang_moi} | Mưa: {so_mua_moi}")
# 4. TẠO TẬP VALIDATION
print("⏳ Đang chuẩn bị tập Validation...")
train_sub, val_sub = train_hybrid.randomSplit([0.8, 0.2], seed=42)
train_sub = train_sub.withColumn("is_val", F.lit(False))
val_sub = val_sub.withColumn("is_val", F.lit(True))
train_with_val = train_sub.union(val_sub)
# 5. KHỞI TẠO VÀ HUẤN LUYỆN XGBOOST
xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="rain_label",
    num_workers=8,
    n_estimators=5000,
    max_depth=8,
    learning_rate=0.1,
    early_stopping_rounds=20,
    validation_indicator_col="is_val",
    scale_pos_weight=0.5
)

print("🚀 Đang tiến hành huấn luyện XGBoost (có Early Stopping)...")
xgb_model = xgb_classifier.fit(train_with_val)
print("✅ Huấn luyện thành công!")
# 6. DỰ ĐOÁN VÀ ĐÁNH GIÁ TRÊN TẬP TEST GỐC
print("🔮 Đang dự đoán trên tập Test...")
predictions = xgb_model.transform(test_data)
# 7. LƯU MÔ HÌNH
model_path = "xgboost_weather_model_cosmotekhongchuan"
xgb_model.write().overwrite().save(model_path)
print(f"💾 Đã lưu mô hình thành công tại thư mục: {model_path}")
# 8. ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH
evaluator = MulticlassClassificationEvaluator(labelCol="rain_label", predictionCol="prediction")
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision_rain = evaluator.evaluate(predictions, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1.0})
recall_rain = evaluator.evaluate(predictions, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1.0})
f1_rain = evaluator.evaluate(predictions, {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: 1.0})

# Tạo nội dung báo cáo
report_content = f"""
==================================================
📊 BẢNG TỔNG HỢP CHỈ SỐ ĐÁNH GIÁ (SAU KHI HYBRID RESAMPLING)
==================================================
✔️ Accuracy  : {accuracy * 100:.2f}%
🎯 Precision : {precision_rain * 100:.2f}%
🎯 Recall    : {recall_rain * 100:.2f}%
🏆 F1-Score  : {f1_rain * 100:.2f}%
==================================================
"""

# In ra màn hình console như bình thường
print(report_content)

# Lưu nội dung vào file text
output_file = "evaluation_results_cosmotekhongchuan.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"📁 Đã xuất kết quả đánh giá thành công ra file: {output_file}")